# Notebook 12: Centralized Gold-Standard Training & Upper-Bound Reference

## 1. Role in Experimental Design
The centralized baseline pools all 100 users' interaction histories on a single server, representing the theoretical **accuracy upper bound** at the cost of complete behavioral surveillance.

In [1]:
import os, sys, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Resolve repository root deterministically from ml/notebooks/
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

DATA_DIR = os.path.join(REPO_ROOT, "ml", "data", "processed")
ARTIFACTS_DIR = os.path.join(REPO_ROOT, "ml", "artifacts")
VIZ_DIR = os.path.join(REPO_ROOT, "ml", "visualizations")
plt.rcParams["figure.dpi"] = 110
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from ml.src.data.dataset import RecSysDataPipeline, BPRDataset
from ml.src.models.content_aware import ContentAwareNeuralRecommender

torch.manual_seed(SEED)
pipeline = RecSysDataPipeline(
    articles_path=os.path.join(DATA_DIR, "articles.json"),
    interactions_path=os.path.join(DATA_DIR, "interactions.csv"),
    embeddings_path=os.path.join(DATA_DIR, "article_embeddings.npy"),
)
model = ContentAwareNeuralRecommender(
    num_users=pipeline.num_users,
    num_items=pipeline.num_items,
    pretrained_embeddings=torch.tensor(pipeline.item_embeddings, dtype=torch.float32),
)
loader = DataLoader(BPRDataset(pipeline.get_bpr_triplets("train", num_negatives=1)), batch_size=64, shuffle=True)
opt = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-5)

losses = []
for ep in range(1, 7):
    model.train()
    el = []
    for u, pos, neg, w in loader:
        opt.zero_grad()
        ps, ns, ue, pe, ne = model(u, pos, neg)
        loss = model.compute_bpr_loss(ps, ns, ue, pe, ne, weights=w)
        loss.backward()
        opt.step()
        el.append(loss.item())
    losses.append(float(np.mean(el)))
    print(f"Centralized Epoch {ep:02d}/06 | Loss = {losses[-1]:.4f}")

with open(os.path.join(ARTIFACTS_DIR, "federated_benchmark.json"), "r") as f:
    bench = json.load(f)
cb = bench["centralized_baseline"]
print(f"\nFull 15-Epoch Centralized Gold Standard -> NDCG@10={cb['ndcg10']:.4f} | HR@10={cb['hr10']:.4f} | MRR={cb['mrr']:.4f}")

Centralized Epoch 01/06 | Loss = 0.5826


Centralized Epoch 02/06 | Loss = 0.4056


Centralized Epoch 03/06 | Loss = 0.3452


Centralized Epoch 04/06 | Loss = 0.2919


Centralized Epoch 05/06 | Loss = 0.2775


Centralized Epoch 06/06 | Loss = 0.2501

Full 15-Epoch Centralized Gold Standard -> NDCG@10=0.1061 | HR@10=0.2400 | MRR=0.0926
